In [0]:
%python
from pyspark.sql.functions import col, regexp_extract, to_json, struct, lit, count, when, concat_ws, date_format, current_timestamp
from pyspark.sql import DataFrame
from functools import reduce
import re

# Definir la ruta de entrada
#ruta_entrada = "abfss://ingestas@stbigdatadev02.dfs.core.windows.net/data/trafico/senalizacion/contadores_zte/parsing_in"

# Definir la ruta de salida (directorio base)
#ruta_salida = "abfss://ingestas@stbigdatadev02.dfs.core.windows.net/data/trafico/senalizacion/contadores_zte/raw"

# Variable de partición
#particion = "2025052007"

# Definir widgets sin valores por defecto
dbutils.widgets.text("ruta_entrada", "", "Ruta de Entrada")
dbutils.widgets.text("ruta_salida", "", "Ruta de Salida")
dbutils.widgets.text("particion", "", "Partición")

# Obtener los valores de los widgets
ruta_entrada = dbutils.widgets.get("ruta_entrada")
ruta_salida = dbutils.widgets.get("ruta_salida")
particion = dbutils.widgets.get("particion")

# Extraer year, month, day, hour de la variable particion
year = particion[:4]  # 2025
month = particion[4:6]  # 05
day = particion[6:8]  # 19
hour = particion[8:10]  # 14

# Columnas comunes
columnas_comunes = ["COLLECTTIME", "SBNID", "MEID", "MENAME", "eNBID", "ENODEBNAME", "NODEBNAME", "CellID", "CellNAME"]

# Función para extraer familias
def extraer_tipo(nombre_archivo):
    match = re.search(r"UMEID_(.+?)_\d{8}", nombre_archivo)
    return match.group(1) if match else None

# Listar todos los archivos y extraer tipos únicos
archivos = dbutils.fs.ls(ruta_entrada)
tipos = set()
for archivo in archivos:
    tipo = extraer_tipo(archivo.name)
    if tipo:
        tipos.add(tipo)

print(f"Tipos de archivos encontrados: {list(tipos)}")

# Lista para almacenar DataFrames
dfs = []

# Leer archivos por tipo
for tipo in tipos:
    df_tipo = spark.read.option("header", True).option("inferSchema", True).csv(f"{ruta_entrada}/*{tipo}*.csv")
    if df_tipo.count() > 0:
        # Agregar columnas comunes faltantes con null
        for columna in columnas_comunes:
            if columna not in df_tipo.columns:
                print(f"Advertencia: {tipo} no tiene la columna {columna}, añadiendo con null")
                df_tipo = df_tipo.withColumn(columna, lit(None).cast("string"))
        
        # Agregar familia
        df_tipo = df_tipo.withColumn("familia", regexp_extract(col("_metadata.file_path"), "UMEID_(.+?)_\d{8}", 1))
        
        # Identificar columnas adicionales
        columnas_adicionales = [c for c in df_tipo.columns if c not in columnas_comunes + ["_metadata", "familia"]]
        
        # Empaquetar columnas adicionales en JSON
        if columnas_adicionales:
            df_tipo = df_tipo.withColumn(
                "contadores",
                to_json(struct([col(c) for c in columnas_adicionales]))
            )
        else:
            df_tipo = df_tipo.withColumn("contadores", to_json(struct([])))
        
        # Seleccionar columnas
        df_tipo = df_tipo.select(columnas_comunes + ["familia", "contadores"]).drop("_metadata")
        
        dfs.append(df_tipo)

# Combinar DataFrames
if dfs:
    df_final = reduce(lambda df1, df2: df1.unionByName(df2), dfs)
    
    # Agregar columnas de partición
    df_final = df_final.withColumn("year", lit(year)) \
                       .withColumn("month", lit(month)) \
                       .withColumn("day", lit(day)) \
                       .withColumn("hour", lit(hour))
    
    # Agregar columna bigdata_close_date (formato yyyy-MM-dd)
    df_final = df_final.withColumn("bigdata_close_date", concat_ws("-", lit(year), lit(month), lit(day)))
    
    # Agregar columna bigdata_ctrl_id (UTC now en formato yyyyMMddHHmmssSSS)
    df_final = df_final.withColumn("bigdata_ctrl_id", date_format(current_timestamp(), "yyyyMMddHHmmssSSS"))
    
    # Mostrar algunos registros
    #df_final.show(20, truncate=False)
    #print(f"Número total de registros: {df_final.count()}")
    
    # Mostrar valores distintos de familia
    #df_final.select("familia").distinct().show(100, truncate=False)
    
    # Contar nulls en columnas comunes
    #df_final.select([count(when(col(c).isNull(), c)).alias(c) for c in #columnas_comunes]).show()
    
    # Guardar en formato Delta, particionado por year, month, day, hour
    df_final.write.format("delta") \
            .mode("append") \
            .partitionBy("year", "month", "day", "hour") \
            .save(ruta_salida)
    
    print(f"Datos guardados en: {ruta_salida}/year={year}/month={month}/day={day}/hour={hour}")
else:
    print("No se encontraron archivos para procesar.")

In [0]:
%python

try:
    # Eliminar la carpeta y todo su contenido
    dbutils.fs.rm(ruta_entrada, recurse=True)
    print(f"La carpeta {ruta_entrada} y todo su contenido han sido eliminados.")

    # Verificar que la carpeta ya no existe
    try:
        dbutils.fs.ls(ruta_entrada)
        print(f"Advertencia: La carpeta {ruta_entrada} aún existe.")
    except:
        print(f"Confirmado: La carpeta {ruta_entrada} no existe.")
        
except Exception as e:
    print(f"Error al eliminar la carpeta {ruta_entrada}: {str(e)}")

In [0]:
import org.apache.spark.sql.functions.{lit, from_unixtime, unix_timestamp}
import org.apache.spark.sql.types.{StringType}
import org.apache.spark.sql.{DataFrame, SaveMode}
import java.util.Date
import java.text.SimpleDateFormat

// Obtener valores desde los widgets de ADF
val pipelineRunId = dbutils.widgets.get("pipelineRunId")
val catalog_control = dbutils.widgets.get("catalog_control")
val starttime_nifi = dbutils.widgets.get("starttime_nifi")
val endtime_nifi = dbutils.widgets.get("endtime_nifi")
val hdfs_path = dbutils.widgets.get("dir_adls")
val original_file_date = dbutils.widgets.get("particion")

// Crear un formatter para los timestamps
val formatter = new SimpleDateFormat("yyyy-MM-dd HH:mm:ss")

// Obtener el timestamp de inicio del proceso Spark
val current = new Date().getTime
val starttime_spark = formatter.format(current)

// Simular controlData con datos de ejemplo, incorporando valores de los widgets
val controlData = Seq(
  (
    "SUCCESS",                     // status
    "Proceso exitoso",            // desc_status
    "ctrl_001",                   // big_data_ctrl_id
    "ingesta_ZTE",                // process_name
    "csv",                        // data_source_type
    "SFTP",                       // data_source_name
    original_file_date,           // original_file_date (desde widget)
    starttime_nifi,               // starttime_nifi (desde widget)
    endtime_nifi,                 // endtime_nifi (desde widget)
    "30",                         // totaltime_nifi
    starttime_spark,              // starttime_spark
    "ETL",                        // process_type
    "1000",                       // original_file_size (StringType)
    "1200",                       // final_file_size (StringType)
    "100",                        // original_row_count (StringType)
    "100",                        // final_row_count (StringType)
    "0",                          // dif_row_count (StringType)
    "1",                          // final_number_of_files (StringType)
    "archivo_final_20250522.csv", // end_file_name
    formatter.format(current),     // insert_data_ctrl_date
    hdfs_path,                    // hdfs_path (desde widget)
    pipelineRunId                 // pipelineRunId (desde widget)
  )
)

// Crear el DataFrame de control
var control_dataframe: DataFrame = spark.createDataFrame(controlData).toDF(
  "status", "desc_status", "big_data_ctrl_id", "process_name", "data_source_type", "data_source_name",
  "original_file_date", "starttime_nifi", "endtime_nifi", "totaltime_nifi", "starttime_spark",
  "process_type", "original_file_size", "final_file_size", "original_row_count", "final_row_count",
  "dif_row_count", "final_number_of_files", "end_file_name", "insert_data_ctrl_date", "hdfs_path",
  "pipelineRunId"
).withColumn("original_file_size", col("original_file_size").cast(StringType))
 .withColumn("final_file_size", col("final_file_size").cast(StringType))
 .withColumn("original_row_count", col("original_row_count").cast(StringType))
 .withColumn("final_row_count", col("final_row_count").cast(StringType))
 .withColumn("dif_row_count", col("dif_row_count").cast(StringType))
 .withColumn("final_number_of_files", col("final_number_of_files").cast(StringType))

// Obtener el timestamp de fin
val end = new Date().getTime
val endtime_spark = formatter.format(end)

// Calcular la duración del proceso Spark
val totaltime_spark = (end - current).toFloat / 1000

// Calcular la duración total del proceso (Spark + NiFi)
var totaltime_process = totaltime_spark
val totaltime_nifi = control_dataframe.select("totaltime_nifi").head().getString(0)
if (totaltime_nifi != "") {
  totaltime_process = totaltime_spark + totaltime_nifi.toFloat
}

// Formatear los timestamps en el DataFrame de control
println("[INFO] SE FORMATEAN LOS TIMESTAMP AL DATAFRAME DE CONTROL")
control_dataframe = control_dataframe
  .withColumn("endtime_spark", lit(endtime_spark))
  .withColumn("totaltime_spark", lit(totaltime_spark))
  .withColumn("totaltime_process", lit(totaltime_process))
  .withColumn("original_file_date", from_unixtime(unix_timestamp(col("original_file_date"), "yyyyMMdd HHmmss"), "yyyy-MM-dd HH:mm:ss"))
  .withColumn("starttime_nifi", from_unixtime(unix_timestamp(col("starttime_nifi"), "yyyyMMdd HHmmss"), "yyyy-MM-dd HH:mm:ss"))
  .withColumn("endtime_nifi", from_unixtime(unix_timestamp(col("endtime_nifi"), "yyyyMMdd HHmmss"), "yyyy-MM-dd HH:mm:ss"))
  .select(
    "status", "desc_status", "big_data_ctrl_id", "process_name", "data_source_type", "data_source_name",
    "original_file_date", "starttime_nifi", "endtime_nifi", "totaltime_nifi", "starttime_spark",
    "endtime_spark", "totaltime_spark", "totaltime_process", "insert_data_ctrl_date", "process_type",
    "original_file_size", "final_file_size", "original_row_count", "final_row_count", "dif_row_count",
    "final_number_of_files", "end_file_name", "hdfs_path", "pipelineRunId"
  )

// Escribir los registros en la tabla de control
control_dataframe.write.mode(SaveMode.Append).option("mergeSchema", "true").saveAsTable(catalog_control)